In [1]:
import scanpy as sc
import magic
import torch
import numpy as np
import pandas as pd

from magic import MAGIC

In [2]:
DATA_FOLDER = '/Users/dm954/Documents/code/mixed_diffusion/data/CITEseq/'

adata = sc.read_h5ad(DATA_FOLDER + 'citeseq_preprocessed.h5ad')

test_indices = np.load(DATA_FOLDER + 'test_indices.npy')
train_indices = np.load(DATA_FOLDER + 'train_indices.npy')

adata_train = adata[train_indices].copy()
adata_test = adata[test_indices].copy()

In [3]:
from sklearn.decomposition import PCA

# Run MAGIC denoising and save denoised embeddings as .pt
PCA_DIM = 25

# Helper to get dense numpy arrays
def to_array(X):
    try:
        if hasattr(X, 'toarray'):
            return X.toarray()
        else:
            return np.asarray(X)
    except Exception:
        return np.asarray(X)

# Fetch raw matrices
X_test_raw = to_array(adata_test.X)

# Run MAGIC denoising
magic_op = MAGIC(n_pca=PCA_DIM)
print('Running MAGIC on train...')
X_test_magic = MAGIC(n_pca=PCA_DIM).fit_transform(X_test_raw)

# Build categorical codes and label encoder (number -> name)
cat_labels = pd.Categorical(adata_test.obs['cell_labels'])
label_test = cat_labels.codes  # integer codes per cell
label_encoder = {str(cat): int(i) for i, cat in enumerate(cat_labels.categories)}

# Apply PCA to reduce X_test_magic to PCA_DIM dimensions
pca = PCA(n_components=PCA_DIM)
X_test_magic = pca.fit_transform(X_test_magic)

X_test_raw = pca.fit_transform(X_test_raw)

print(X_test_magic.shape, X_test_raw.shape)

test_magic = {
    'x_denoised': torch.tensor(X_test_magic, dtype=torch.float32),
    'x_true': torch.tensor(X_test_raw, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_test),
    'data_config': { 'label_encoder': label_encoder }
}

out_test = 'magic/denoising_results.pt'
torch.save(test_magic, out_test)
print(f"Saved MAGIC denoised file: '{out_test}'")

Running MAGIC on train...
Calculating MAGIC...
  Running MAGIC on 1000 cells and 3000 genes.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 0.35 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.08 seconds.
    Calculating affinities...
    Calculated affinities in 0.68 seconds.
  Calculated graph and diffusion operator in 1.11 seconds.
  Calculating imputation...
  Calculated imputation in 0.08 seconds.
Calculated MAGIC in 1.20 seconds.
(1000, 25) (1000, 25)
Saved MAGIC denoised file: 'magic/denoising_results.pt'


In [4]:
adata_test.layers['counts']

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 289720 stored elements and shape (1000, 3000)>

In [5]:
from pyALRA import alra, normalize_data, choose_k

# Load your data into an AnnData object

count_matrix = adata_test.layers['counts'].toarray()
normalized_count_matrix = normalize_data(count_matrix)

# Determine the optimal k
k = choose_k(normalized_count_matrix)

# Apply ALRA
embeddings = alra(normalized_count_matrix, k['k'])['A_norm_rank_k_cor_sc']
print(embeddings.shape)
print(k)

# Apply PCA to reduce embeddings to PCA_DIM dimensions
pca = PCA(n_components=PCA_DIM)
X_alra_denoised = pca.fit_transform(embeddings)
X_alra_raw = pca.fit_transform(to_array(adata_test.X))

print(X_alra_denoised.shape, X_alra_raw.shape)

# Build categorical codes and label encoder
cat_labels_alra = pd.Categorical(adata_test.obs['cell_labels'])
label_alra = cat_labels_alra.codes

alra_results = {
    'x_denoised': torch.tensor(X_alra_denoised, dtype=torch.float32),
    'x_true': torch.tensor(X_alra_raw, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_alra),
    'data_config': {'label_encoder': label_encoder}
}

out_alra = 'alra/denoising_results.pt'
torch.save(alra_results, out_alra)
print(f"Saved ALRA denoised file: '{out_alra}'")


Read matrix with 1000 cells and 3000 genes
Find the 0.001 quantile of each gene
Sweep
Scaling all except for 640 columns


/opt/homebrew/anaconda3/envs/mixed_diffusion_baselines/lib/python3.10/site-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/anaconda3/envs/mixed_diffusion_baselines/lib/python3.10/site-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/homebrew/anaconda3/envs/mixed_diffusion_baselines/lib/python3.10/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/opt/homebrew/anaconda3/envs/mixed_diffusion_baselines/lib/python3.10/site-packages/pyALRA/core.py:185: RuntimeWarning: invalid value encountered in divide
  mu_1 = np.sum(A_norm_rank_k_cor, axis=0) / np.sum(A_norm_rank_k_cor != 0, axis=0)
/opt/homebrew/anaconda3/envs/mixed_diffusion_baselines/lib/python3.10/site-packages/pyALRA/core.py:186: RuntimeWarni

0.00% of the values became negative in the scaling process and were set to zero
The matrix went from 9.66% nonzero to 16.09% nonzero
(1000, 3000)
{'k': np.int64(36), 'num_of_sds': array([ 9.96577734e+03,  3.16631348e+03,  7.85293945e+02,  8.96090637e+02,
        1.86100143e+02,  1.41077606e+02,  7.25540085e+01,  1.92874985e+02,
        4.81406441e+01,  2.34587765e+01,  6.39694023e+01,  1.21301575e+02,
        6.28724289e+01,  1.21222086e+01,  4.11087799e+01,  3.04792857e+00,
        5.62177229e+00,  1.74063148e+01,  1.39748650e+01,  4.45921850e+00,
        1.50903044e+01,  9.08674431e+00,  4.53238249e+00,  4.77272415e+00,
        3.38033438e+00,  1.97104824e+00,  2.15842605e+00,  1.02922440e+01,
        8.55380058e+00,  1.73866749e+00,  5.05351387e-02,  1.65602636e+00,
        1.70926106e+00, -9.39209104e-01,  4.02169704e+00,  6.01299095e+00,
        6.54801190e-01,  2.09728456e+00, -1.24399567e+00,  4.85023975e-01,
        3.66465020e+00,  2.17496261e-01, -9.16193008e-01,  2.69662237e

In [6]:
# Run sklearn NMF on test data and save results (using non-negative matrix factorization)
import os
import numpy as np
from sklearn.decomposition import NMF, PCA

print(f"Running sklearn NMF on adata_test with n_components={PCA_DIM}...")

# Use counts layer if available, otherwise adata_test.X
X_counts = to_array(adata_test.layers['counts']) if 'counts' in adata_test.layers else to_array(adata_test.X)
# Ensure non-negative
X_counts = np.clip(X_counts, a_min=0.0, a_max=None)
# Apply log1p transform (preserve non-negativity)
X_counts_log = np.log1p(X_counts)

# Fit NMF on log-transformed counts
nmf_model = NMF(n_components=PCA_DIM, init='nndsvda', random_state=0)
W = nmf_model.fit_transform(X_counts_log)  # cells x components
H = nmf_model.components_

# Prepare x_true: apply log1p to raw and reduce with PCA for comparability
X_raw = to_array(adata_test.X)
X_raw_log = np.log1p(X_raw)
pca_nmf = PCA(n_components=PCA_DIM)


# Build categorical codes and label encoder
cat_labels_nmf = pd.Categorical(adata_test.obs['cell_labels'])
label_nmf = cat_labels_nmf.codes
if 'label_encoder' not in globals():
    label_encoder = {str(cat): int(i) for i, cat in enumerate(cat_labels_nmf.categories)}

nmf_results = {
    'x_denoised': torch.tensor(W, dtype=torch.float32),
    'x_true': torch.tensor(W, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_nmf),
    'data_config': {'label_encoder': label_encoder}
}

out_nmf = 'nmf/denoising_results.pt'
# ensure output dir
out_dir = os.path.dirname(out_nmf)
if out_dir and not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)

torch.save(nmf_results, out_nmf)
print(f"Saved sklearn NMF denoised file: '{out_nmf}'")

Running sklearn NMF on adata_test with n_components=25...
Saved sklearn NMF denoised file: 'nmf/denoising_results.pt'


/opt/homebrew/anaconda3/envs/mixed_diffusion_baselines/lib/python3.10/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/var/folders/mx/8nbm269n3hv_3jv51d0k5jr40000gp/T/ipykernel_7567/2772938198.py:22: RuntimeWarning: invalid value encountered in log1p
  X_raw_log = np.log1p(X_raw)


In [10]:
from scipy.sparse import issparse
def knn_moments(adata, n_neighbors=30, n_pcs=50, layer=None):
    """
    Smooth expression data using kNN neighbors' mean and variance.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data matrix
    n_neighbors : int
        Number of neighbors to use for smoothing
    n_pcs : int
        Number of PCs to use for kNN computation
    layer : str or None
        Layer to smooth. If None, uses adata.X
        
    Returns
    -------
    Modifies adata in place:
    - adata.layers["Ms"] : smoothed mean expression (kNN-averaged)
    - adata.layers["Vs"] : smoothed variance (kNN-averaged)
    """
    
    # Compute neighbors if not already computed
    print("Computing neighbors...")
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)
    
    # Get the data to smooth
    if layer is None:
        X = adata.X
    else:
        X = adata.layers[layer]
    
    # Convert to dense if sparse
    if issparse(X):
        X = X.toarray()
    
    # Get neighbor indices from the distances/connectivities
    connectivities = adata.obsp["connectivities"]
    
    n_obs, n_vars = X.shape
    
    # Initialize smoothed mean and variance
    Ms = np.zeros_like(X, dtype=np.float32)
    Vs = np.zeros_like(X, dtype=np.float32)
    
    print("Computing kNN moments...")
    
    # For each cell, compute mean and variance over its neighbors
    for i in range(n_obs):
        # Get neighbor indices (including self)
        neighbors = connectivities[i].nonzero()[1]
        
        if len(neighbors) > 0:
            # Get neighbor expression values
            neighbor_expr = X[neighbors, :]
            
            # Compute mean over neighbors
            Ms[i, :] = np.mean(neighbor_expr, axis=0)
            
            # Compute variance over neighbors
            Vs[i, :] = np.var(neighbor_expr, axis=0)
        
        if (i + 1) % 1000 == 0:
            print(f"  Processed {i + 1}/{n_obs} cells")
    
    # Store results
    adata.layers["Ms"] = Ms
    adata.layers["Vs"] = Vs
    
    print("Done!")
    return adata

In [13]:
import os
import numpy as np
from sklearn.decomposition import NMF, PCA

print(f"Running adata_test with n_components={PCA_DIM}...")

for k in [3, 5, 7, 10, 15, 30]:

    knn_moments(adata_test, n_neighbors=k, n_pcs=PCA_DIM)
    print(adata_test.layers)

    W = adata_test.layers["Ms"].copy() 


    knn_results = {
        'x_denoised': torch.tensor(W, dtype=torch.float32),
        'x_true': torch.tensor(W, dtype=torch.float32),
        'x_denoised_labels': torch.tensor(label_nmf),
        'data_config': {'label_encoder': label_encoder}
    }

    out_knn = f"knn_{k}/denoising_results.pt"
    # ensure output dir
    out_dir = os.path.dirname(out_knn)
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    torch.save(knn_results, out_knn)
    print(f"Saved sklearn NMF denoised file: '{out_knn}'")

Running adata_test with n_components=25...
Computing neighbors...
Computing kNN moments...
  Processed 1000/1000 cells
Done!
Layers with keys: counts, Ms, Vs
Saved sklearn NMF denoised file: 'knn_3/denoising_results.pt'
Computing neighbors...
Computing kNN moments...
  Processed 1000/1000 cells
Done!
Layers with keys: counts, Ms, Vs
Saved sklearn NMF denoised file: 'knn_5/denoising_results.pt'
Computing neighbors...
Computing kNN moments...
  Processed 1000/1000 cells
Done!
Layers with keys: counts, Ms, Vs
Saved sklearn NMF denoised file: 'knn_7/denoising_results.pt'
Computing neighbors...
Computing kNN moments...
  Processed 1000/1000 cells
Done!
Layers with keys: counts, Ms, Vs
Saved sklearn NMF denoised file: 'knn_10/denoising_results.pt'
Computing neighbors...
Computing kNN moments...
  Processed 1000/1000 cells
Done!
Layers with keys: counts, Ms, Vs
Saved sklearn NMF denoised file: 'knn_15/denoising_results.pt'
Computing neighbors...
Computing kNN moments...
  Processed 1000/1000 

In [9]:
import os
import numpy as np
from sklearn.decomposition import NMF, PCA

print(f"Running adata_test with just PCA")


# Apply PCA to reduce X_test_magic to PCA_DIM dimensions
pca = PCA(n_components=PCA_DIM)
W = pca.fit_transform(adata_test.X)
pca_results = {
    'x_denoised': torch.tensor(W, dtype=torch.float32),
    'x_true': torch.tensor(W, dtype=torch.float32),
    'x_denoised_labels': torch.tensor(label_nmf),
    'data_config': {'label_encoder': label_encoder}
}

out_pca = f"pca/denoising_results.pt"
# ensure output dir
out_dir = os.path.dirname(out_pca)
if out_dir and not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)

torch.save(pca_results, out_pca)
print(f"Saved sklearn NMF denoised file: '{out_pca}'")

Running adata_test with just PCA
Saved sklearn NMF denoised file: 'pca/denoising_results.pt'
